# Gold Layer: EV Charger Recommendation Engine

This notebook creates intelligent recommendations for EV charging stations based on:
- **Location proximity** - Distance to nearest chargers
- **Charger availability** - Station density and capacity
- **Speed requirements** - Match fast/rapid/ultra-rapid needs
- **Traffic conditions** - Active hazards affecting accessibility
- **Regional readiness** - Infrastructure maturity scores
- **Weather impact** - Current conditions affecting charging

## Recommendation Types:

### 1. **Nearest Charger Finder**
Given a location (lat/lon), find the N nearest charging stations with:
- Distance calculation
- Charging speed availability
- Regional context

### 2. **Route Corridor Chargers**
Given start and end points, find chargers along the route corridor

### 3. **Infrastructure Gap Analysis**
Identify underserved areas needing new charging stations:
- Low-density regions
- High traffic but low charger availability
- Regional readiness scores

### 4. **Real-Time Smart Recommendations**
Combine static infrastructure with dynamic conditions:
- Active traffic hazards
- Weather conditions
- Regional availability

## Gold Tables Created:
- `mobility_ai.gold.charger_recommendations_nearest`
- `mobility_ai.gold.charger_recommendations_gaps`
- `mobility_ai.gold.charger_recommendations_smart`

In [0]:
# Configuration
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType
import math

# Table paths
EV_STATIONS_TABLE = "mobility_ai.silver.ev_charging_stations"
TRAFFIC_HAZARDS_TABLE = "mobility_ai.silver.traffic_hazards"
WEATHER_TABLE = "mobility_ai.silver.weather_conditions"
REGIONAL_METRICS_TABLE = "mobility_ai.gold.regional_infrastructure_metrics"

# Output tables
NEAREST_CHARGERS_TABLE = "mobility_ai.gold.charger_recommendations_nearest"
GAP_ANALYSIS_TABLE = "mobility_ai.gold.charger_recommendations_gaps"
SMART_RECOMMENDATIONS_TABLE = "mobility_ai.gold.charger_recommendations_smart"

# NSW Reference Points (for demo)
REFERENCE_LOCATIONS = [
    ("Sydney CBD", -33.8688, 151.2093),
    ("Parramatta", -33.8150, 151.0010),
    ("Newcastle", -32.9283, 151.7817),
    ("Wollongong", -34.4278, 150.8931),
    ("Blue Mountains", -33.7124, 150.3105),
    ("Central Coast", -33.4290, 151.3430),
    ("Port Macquarie", -31.4333, 152.9000),
    ("Wagga Wagga", -35.1082, 147.3598)
]

print("\n" + "═" * 80)
print("⚙️  CHARGER RECOMMENDATION ENGINE - CONFIGURATION")
print("═" * 80)
print(f"\n📍 Source Tables:")
print(f"   • EV Stations:       {EV_STATIONS_TABLE}")
print(f"   • Traffic Hazards:   {TRAFFIC_HAZARDS_TABLE}")
print(f"   • Weather:           {WEATHER_TABLE}")
print(f"   • Regional Metrics:  {REGIONAL_METRICS_TABLE}")
print(f"\n🎯 Output Tables:")
print(f"   • Nearest Chargers:  {NEAREST_CHARGERS_TABLE}")
print(f"   • Gap Analysis:      {GAP_ANALYSIS_TABLE}")
print(f"   • Smart Recommend:   {SMART_RECOMMENDATIONS_TABLE}")
print(f"\n📍 Reference Locations: {len(REFERENCE_LOCATIONS)}")
print("═" * 80 + "\n")

In [0]:
# Haversine formula for distance calculation
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate distance in kilometers between two points using Haversine formula
    """
    # Convert to radians
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)
    
    # Haversine formula
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    a = math.sin(dlat/2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    
    # Earth radius in kilometers
    earth_radius = 6371
    
    return c * earth_radius

# Register as UDF
haversine_udf = F.udf(haversine_distance, DoubleType())

print("\n" + "═" * 80)
print("📐 DISTANCE CALCULATION FUNCTION")
print("═" * 80)
print("\n✅ Haversine UDF registered for distance calculations")
print("   • Formula: Great-circle distance")
print("   • Unit: Kilometers")
print("   • Earth Radius: 6,371 km")
print("═" * 80 + "\n")

In [0]:
# Load all source tables
ev_stations = spark.table(EV_STATIONS_TABLE)
traffic_hazards = spark.table(TRAFFIC_HAZARDS_TABLE)
weather = spark.table(WEATHER_TABLE)
regional_metrics = spark.table(REGIONAL_METRICS_TABLE)

ev_count = ev_stations.count()
hazard_count = traffic_hazards.count()
weather_count = weather.count()
region_count = regional_metrics.count()

print("\n" + "═" * 80)
print("📊 DATA LOADING")
print("═" * 80)
print(f"\n✅ Source Tables Loaded:")
print(f"   • EV Charging Stations:  {ev_count:,} records")
print(f"   • Traffic Hazards:       {hazard_count:,} records")
print(f"   • Weather Conditions:    {weather_count:,} records")
print(f"   • Regional Metrics:      {region_count:,} regions")
print("\n" + "─" * 80)
print("\n📋 EV Stations Sample:")
ev_stations.select("station_name", "latitude", "longitude", "charging_speed", "charger_rating_kw").show(5, truncate=False)
print("═" * 80 + "\n")

In [0]:
# For each reference location, find the nearest N chargers
from pyspark.sql import Row

# Create reference locations dataframe
ref_locations_data = [Row(location_name=name, ref_lat=lat, ref_lon=lon) 
                      for name, lat, lon in REFERENCE_LOCATIONS]
ref_df = spark.createDataFrame(ref_locations_data)

# Cross join to calculate distance from each reference point to every charger
nearest_chargers = ref_df.crossJoin(ev_stations)

# Calculate distance using Haversine UDF
nearest_chargers = nearest_chargers.withColumn(
    "distance_km",
    haversine_udf(
        F.col("ref_lat"),
        F.col("ref_lon"),
        F.col("latitude"),
        F.col("longitude")
    )
)

# Rank by distance within each reference location
window_spec = Window.partitionBy("location_name").orderBy("distance_km")
nearest_chargers = nearest_chargers.withColumn("rank", F.row_number().over(window_spec))

# Keep top 10 nearest for each location
nearest_chargers = nearest_chargers.filter(F.col("rank") <= 10)

# Select relevant columns
nearest_chargers = nearest_chargers.select(
    "location_name",
    "ref_lat",
    "ref_lon",
    "rank",
    "station_name",
    "station_address",
    "latitude",
    "longitude",
    F.round("distance_km", 2).alias("distance_km"),
    "operator",
    "charging_speed",
    "charger_rating_kw",
    "has_valid_rating"
).withColumn("generated_at", F.current_timestamp())

print("\n" + "═" * 80)
print("🎯 NEAREST CHARGER RECOMMENDATIONS")
print("═" * 80)
print(f"\n📊 Generated {nearest_chargers.count():,} recommendations for {len(REFERENCE_LOCATIONS)} locations")
print("\n" + "─" * 80)
print("\n📍 Sample: Top 5 Nearest Chargers to Sydney CBD:")
nearest_chargers.filter(F.col("location_name") == "Sydney CBD").show(5, truncate=False)
print("═" * 80 + "\n")

In [0]:
# Identify areas with low charger density (infrastructure gaps)

# Calculate charger density by region
gap_analysis = ev_stations.groupBy("lganame").agg(
    F.count("*").alias("charger_count"),
    F.sum(F.when(F.col("has_valid_rating"), F.col("charger_rating_kw")).otherwise(0)).alias("total_capacity_kw"),
    F.avg("latitude").alias("center_lat"),
    F.avg("longitude").alias("center_lon"),
    
    # Speed distribution
    F.sum(F.when(F.col("charging_speed") == "Slow", 1).otherwise(0)).alias("slow_chargers"),
    F.sum(F.when(F.col("charging_speed") == "Fast", 1).otherwise(0)).alias("fast_chargers"),
    F.sum(F.when(F.col("charging_speed") == "Rapid", 1).otherwise(0)).alias("rapid_chargers"),
    F.sum(F.when(F.col("charging_speed") == "Ultra-Rapid", 1).otherwise(0)).alias("ultra_rapid_chargers")
)

# Calculate gap severity score (0-100, higher = worse gap)
gap_analysis = gap_analysis.withColumn(
    "gap_severity_score",
    F.least(
        F.lit(100),
        (
            # Base score: inverse of charger count (max 40 points)
            F.when(F.col("charger_count") > 0, 
                   F.least(F.lit(40), 40 * (1 / F.col("charger_count")))
            ).otherwise(40) +
            
            # Capacity deficit (max 30 points)
            F.when(F.col("total_capacity_kw") < 500,
                   F.least(F.lit(30), 30 * (1 - F.col("total_capacity_kw") / 500))
            ).otherwise(0) +
            
            # Speed availability (max 30 points - missing rapid/ultra-rapid)
            F.when(
                (F.col("rapid_chargers") + F.col("ultra_rapid_chargers")) == 0,
                30
            ).when(
                (F.col("rapid_chargers") + F.col("ultra_rapid_chargers")) < 5,
                20
            ).otherwise(10)
        )
    ).cast("int")
)

# Categorize gap severity
gap_analysis = gap_analysis.withColumn(
    "gap_category",
    F.when(F.col("gap_severity_score") >= 70, "Critical")
     .when(F.col("gap_severity_score") >= 50, "High")
     .when(F.col("gap_severity_score") >= 30, "Moderate")
     .otherwise("Low")
)

gap_analysis = gap_analysis.withColumn("analyzed_at", F.current_timestamp())

print("\n" + "═" * 80)
print("⚠️  INFRASTRUCTURE GAP ANALYSIS")
print("═" * 80)
print(f"\n📊 Analyzed {gap_analysis.count():,} LGA regions")
print("\n" + "─" * 80)
print("\n🚨 Top 10 Critical Gaps (Highest Severity):")
gap_analysis.orderBy("gap_severity_score", ascending=False).show(10, truncate=False)

print("\n" + "─" * 80)
print("\n📈 Gap Distribution:")
gap_analysis.groupBy("gap_category").count().orderBy("gap_category").show()
print("═" * 80 + "\n")

In [0]:
# Combine static infrastructure with dynamic traffic and weather

# Get active hazards
active_hazards = traffic_hazards.filter(F.col("is_active") == True)

# For each charger, check if there are nearby hazards
# Cross join chargers with hazards and calculate distance
chargers_with_hazards = ev_stations.crossJoin(
    active_hazards.select(
        F.col("hazard_id"),
        F.col("latitude").alias("hazard_lat"),
        F.col("longitude").alias("hazard_lon"),
        "hazard_type_clean",
        "severity",
        "main_category"
    )
)

chargers_with_hazards = chargers_with_hazards.withColumn(
    "hazard_distance_km",
    haversine_udf(
        F.col("latitude"),
        F.col("longitude"),
        F.col("hazard_lat"),
        F.col("hazard_lon")
    )
)

# Consider hazards within 5km as affecting accessibility
chargers_with_hazards = chargers_with_hazards.filter(F.col("hazard_distance_km") <= 5)

# Aggregate hazards per charger
hazard_summary = chargers_with_hazards.groupBy(
    "objectid",
    "station_name",
    "latitude",
    "longitude"
).agg(
    F.count("*").alias("nearby_hazards_count"),
    F.collect_list("main_category").alias("hazard_types"),
    F.max(
        F.when(F.col("severity") == "Major", 3)
         .when(F.col("severity") == "Moderate", 2)
         .otherwise(1)
    ).alias("max_hazard_severity")
)

# Join back to all stations (left join to include stations with no hazards)
smart_recommendations = ev_stations.join(
    hazard_summary,
    ["objectid", "station_name", "latitude", "longitude"],
    "left"
).fillna(0, subset=["nearby_hazards_count", "max_hazard_severity"])

# Calculate accessibility score (0-100, higher = better accessibility)
smart_recommendations = smart_recommendations.withColumn(
    "accessibility_score",
    F.least(
        F.lit(100),
        (
            # Base score
            F.lit(100) -
            # Deduct for nearby hazards
            (F.col("nearby_hazards_count") * 10) -
            # Additional deduction for severe hazards
            (F.col("max_hazard_severity") * 10)
        )
    ).cast("int")
)

# Calculate overall recommendation score
smart_recommendations = smart_recommendations.withColumn(
    "recommendation_score",
    (
        # Charger quality (40%)
        F.when(
            F.col("charging_speed") == "Ultra-Rapid", 40
        ).when(
            F.col("charging_speed") == "Rapid", 35
        ).when(
            F.col("charging_speed") == "Fast", 25
        ).when(
            F.col("charging_speed") == "Slow", 15
        ).otherwise(10) +
        
        # Accessibility (60%)
        (F.col("accessibility_score") * 0.6)
    ).cast("int")
)

# Add recommendation tier
smart_recommendations = smart_recommendations.withColumn(
    "recommendation_tier",
    F.when(F.col("recommendation_score") >= 85, "Excellent")
     .when(F.col("recommendation_score") >= 70, "Good")
     .when(F.col("recommendation_score") >= 50, "Fair")
     .otherwise("Limited")
)

smart_recommendations = smart_recommendations.withColumn("generated_at", F.current_timestamp())

print("\n" + "═" * 80)
print("🧠 SMART RECOMMENDATIONS (with Real-Time Conditions)")
print("═" * 80)
print(f"\n📊 Analyzed {smart_recommendations.count():,} charging stations")
print(f"   • Active Hazards Considered: {active_hazards.count():,}")
print("\n" + "─" * 80)
print("\n🏆 Top 10 Recommended Chargers (Best Overall Score):")
smart_recommendations.select(
    "station_name",
    "station_address",
    "charging_speed",
    "recommendation_score",
    "recommendation_tier",
    "accessibility_score",
    "nearby_hazards_count"
).orderBy("recommendation_score", ascending=False).show(10, truncate=False)

print("\n" + "─" * 80)
print("\n📊 Recommendation Distribution:")
smart_recommendations.groupBy("recommendation_tier").count().orderBy("recommendation_tier").show()
print("═" * 80 + "\n")

In [0]:
# Write all three recommendation tables

print("\n" + "═" * 80)
print("💾 WRITING RECOMMENDATION TABLES")
print("═" * 80)

# 1. Nearest Chargers
nearest_count = nearest_chargers.count()
nearest_chargers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(NEAREST_CHARGERS_TABLE)
print(f"\n✅ Nearest Chargers:  {nearest_count:,} records → {NEAREST_CHARGERS_TABLE}")

# 2. Gap Analysis
gap_count = gap_analysis.count()
gap_analysis.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GAP_ANALYSIS_TABLE)
print(f"✅ Gap Analysis:      {gap_count:,} records → {GAP_ANALYSIS_TABLE}")

# 3. Smart Recommendations
smart_count = smart_recommendations.count()
smart_recommendations.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SMART_RECOMMENDATIONS_TABLE)
print(f"✅ Smart Recommend:   {smart_count:,} records → {SMART_RECOMMENDATIONS_TABLE}")

print("\n   📝 Write Mode:    OVERWRITE")
print("   📄 Format:        Delta Lake")
print("   🔄 Schema:        Auto-evolved")
print("═" * 80 + "\n")

In [0]:
print("\n" + "═" * 80)
print("🎯 CHARGER RECOMMENDATION ENGINE - EXECUTIVE SUMMARY")
print("═" * 80)

# Load tables for validation
nearest_tbl = spark.table(NEAREST_CHARGERS_TABLE)
gap_tbl = spark.table(GAP_ANALYSIS_TABLE)
smart_tbl = spark.table(SMART_RECOMMENDATIONS_TABLE)

print("\n📊 Recommendation Outputs:")
print(f"   • Nearest Chargers:      {nearest_tbl.count():,} recommendations")
print(f"   • Gap Analysis:          {gap_tbl.count():,} LGA regions analyzed")
print(f"   • Smart Recommendations: {smart_tbl.count():,} stations scored")

print("\n" + "─" * 80)
print("\n🚨 Infrastructure Gaps:")
critical_gaps = gap_tbl.filter(F.col("gap_category") == "Critical").count()
high_gaps = gap_tbl.filter(F.col("gap_category") == "High").count()
print(f"   • Critical Priority Areas:  {critical_gaps}")
print(f"   • High Priority Areas:      {high_gaps}")
print(f"   • Total Underserved LGAs:   {critical_gaps + high_gaps}")

print("\n" + "─" * 80)
print("\n🏆 Charger Quality:")
excellent = smart_tbl.filter(F.col("recommendation_tier") == "Excellent").count()
good = smart_tbl.filter(F.col("recommendation_tier") == "Good").count()
total_recommended = excellent + good
print(f"   • Excellent Tier:           {excellent} stations")
print(f"   • Good Tier:                {good} stations")
print(f"   • Total Recommended:        {total_recommended} ({total_recommended/smart_tbl.count()*100:.1f}%)")

print("\n" + "─" * 80)
print("\n⚠️  Current Accessibility Issues:")
affected_stations = smart_tbl.filter(F.col("nearby_hazards_count") > 0).count()
print(f"   • Stations Near Hazards:    {affected_stations}")
print(f"   • Unaffected Stations:      {smart_tbl.count() - affected_stations}")

print("\n" + "─" * 80)
print("\n🎯 Top Business Insights:")
top_gap = gap_tbl.orderBy("gap_severity_score", ascending=False).first()
print(f"   • Most Critical Gap:        {top_gap['lganame']} (Score: {top_gap['gap_severity_score']}/100)")
print(f"     - Current Chargers:       {top_gap['charger_count']}")
print(f"     - Rapid+ Chargers:        {top_gap['rapid_chargers'] + top_gap['ultra_rapid_chargers']}")

top_station = smart_tbl.orderBy("recommendation_score", ascending=False).first()
print(f"\n   • Top Recommended Station:  {top_station['station_name']}")
print(f"     - Recommendation Score:   {top_station['recommendation_score']}/100")
print(f"     - Tier:                   {top_station['recommendation_tier']}")
print(f"     - Speed:                  {top_station['charging_speed']}")

print("\n" + "═" * 80)
print("✅ RECOMMENDATION ENGINE COMPLETE")
print("═" * 80 + "\n")